In [2]:
import torch
from d2l import torch as d2l

## 1. Introduction to Language Models

The primary goal of a **language model** is to estimate the joint probability of an entire text sequence composed of discrete tokens $x_1, x_2, \dots, x_T$ (such as words or characters):


$$P(x_1, x_2, \dots, x_T)$$



* **Generative Capability:** An ideal language model can generate natural text by sequentially drawing tokens from the conditional distribution $x_t \sim P(x_t \mid x_{t-1}, \dots, x_1)$.


* **Practical Utility:** Even in limited forms, language models resolve ambiguity in speech recognition (e.g., distinguishing "to recognize speech" from "to wreck a nice beach") and evaluate text naturalness or likelihood in document summarization.



---

## 2. Learning Language Models via the Chain Rule

To model a document or token sequence, we apply basic probability rules (the chain rule) to factorize the joint probability:


$$P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})$$



For example, a four-word sequence like $P(\text{deep}, \text{learning}, \text{is}, \text{fun})$ expands to:


$$P(\text{deep}) P(\text{learning} \mid \text{deep}) P(\text{is} \mid \text{deep}, \text{learning}) P(\text{fun} \mid \text{deep}, \text{learning}, \text{is})$$



---

## 3. Markov Models and $n$-grams

Because conditioning on an entire historical context becomes unmanageable, we apply **Markov assumptions** to restrict how far back the model looks:

* **Unigram ($n=1$):** Assumes independence, so $P(x_1, x_2, x_3, x_4) = P(x_1)P(x_2)P(x_3)P(x_4)$.


* **Bigram ($n=2$, First-Order Markov):** Conditioned only on the immediate predecessor, $P(x_t \mid x_1, \dots, x_1) = P(x_t \mid x_{t-1})$.


* **Trigram ($n=3$, Second-Order Markov):** Conditioned on the previous two words.



### Estimating Word Frequencies

Using a large corpus (like Wikipedia or web text), probabilities are estimated using relative word frequencies:

* **Unigram Estimate:** $\hat{P}(x) = \frac{n(x)}{n}$ (where $n(x)$ is the count of occurrences and $n$ is total words).


* **Bigram Estimate:** $\hat{P}(x' \mid x) = \frac{n(x, x')}{n(x)}$ (the count of consecutive word pairs divided by the count of the single word).



---

## 4. The Sparsity Problem & Laplace Smoothing

As $n$-grams grow longer (trigrams and beyond), many plausible word combinations will **never appear** in the training dataset, resulting in zero counts and breaking the language model.

To fix zero-probability issues for unseen words or sequences, we use **Laplace smoothing**, which adds a small constant to all counts:


$$\hat{P}(x) = \frac{n(x) + \epsilon_1 / m}{n + \epsilon_1}$$



$$\hat{P}(x' \mid x) = \frac{n(x, x') + \epsilon_2 \hat{P}(x')}{n(x) + \epsilon_2}$$



* **Hyperparameters ($\epsilon_1, \epsilon_2, \epsilon_3$):** Control the smoothing strength. For instance, if $\epsilon_1 = 0$, no smoothing is applied; as $\epsilon_1$ approaches infinity, $\hat{P}(x)$ approaches a uniform probability distribution $1/m$ (where $m$ is the number of unique words).



### Limitations of Count-Based Models

Despite smoothing, traditional $n$-gram models face deep limitations:

1. Many $n$-grams occur so rarely that count-based smoothing is insufficient.


2. They require storing massive frequency tables.


3. They completely ignore semantic similarity (e.g., treating "cat" and "feline" as entirely unrelated contexts).



These flaws motivate the shift toward **neural network-based language models**.

Based on the provided pages from *Dive into Deep Learning* (Sections 9.3.2/9.3.3), here is a detailed explanation of how we measure the quality of a language model using **cross-entropy** and **perplexity**, followed by how we partition sequences for training.

---

## 1. Evaluating Language Models: The Problem with Raw Likelihood

To assess how good a language model is, we want to see how surprising its text predictions are. While we could measure the likelihood of an entire sequence, it is hard to interpret and compare because **shorter sequences are inherently much more likely than longer ones**. For example, evaluating a model on a massive book like *War and Peace* will naturally yield a much smaller likelihood than a short novella like *The Little Prince*. What we need is an **average** per token.

---

## 2. Cross-Entropy Loss

Drawing from information theory, if we want to compress text by predicting the next token given the current context, a better model allows us to predict more accurately and spend fewer bits on compression.

We measure this using the **cross-entropy loss averaged over all $n$ tokens** of a sequence:

$$\frac{1}{n} \sum_{t=1}^{n} -\log P(x_t \mid x_{t-1}, \dots, x_1)$$



* **$P$:** Given by the language model.


* **$x_t$:** The actual token observed at time step $t$ from the sequence.


* Averaging by $n$ makes performance on documents of different lengths directly comparable.



---

## 3. Perplexity

For historical reasons, natural language processing scientists prefer using a quantity called **perplexity**, which is defined as the exponential of the cross-entropy loss:

$$\exp\left(\frac{1}{n} \sum_{t=1}^{n} -\log P(x_t \mid x_{t-1}, \dots, x_1)\right)$$



Perplexity can be understood as the reciprocal of the geometric mean of the number of real choices we have when deciding which token to pick next.

### Interpreting Perplexity Values:

* **Best-case scenario ($1$):** The model always perfectly estimates the probability of the target token as $1$.


* **Worst-case scenario ($\infty$):** The model always predicts the probability of the target token as $0$, resulting in positive infinity perplexity.


* **Baseline scenario (Vocabulary Size):** The model predicts a uniform distribution over all available tokens in the vocabulary. In this case, the perplexity equals the **number of unique tokens in the vocabulary** (which is equivalent to storing the sequence without any compression). Any useful model must beat this baseline.



---

## 4. Partitioning Sequences for Neural Training

When training neural language models, we need to handle minibatches of text sequences with predefined lengths ($n$).

* **The Goal:** Predict the next token based on tokens seen so far, meaning the targets (labels) are the original sequence **shifted by one token** ($x_{t+1}$ for any input sequence $x_t$ of length $n$).


* **Introducing Randomness ($d$):** To iterate over a dataset efficiently across epochs, we can introduce randomness by discarding a randomly sampled number of tokens ($d \in [0, n]$) at the beginning of each epoch.


* The remaining sequence is then partitioned into consecutive subsequences of length $n$, which serve as input and target minibatches for the neural language model.

When training neural language models, we can't feed an entire massive book into the network all at once. Instead, we need to chop it up into manageable chunks (minibatches of fixed length $n$) so the neural network can process them efficiently.

Here is a step-by-step breakdown of how **sequence partitioning** works using a concrete example.

---

### Step 1: The Raw Corpus and Parameters

Suppose our entire dataset (corpus) is a single text sequence containing $T = 12$ tokens:

> `["the", "time", "machine", "by", "h", "g", "wells", "and", "other", "tales", "is", "here"]`

We decide on two parameters:

1. **Sequence length ($n$ = 5):** Each training subsequence will contain 5 tokens.


2. **Random offset ($d$):** To ensure the model sees different word combinations across different training epochs, we randomly discard a number of tokens $d \in [0, n]$ at the very beginning of the dataset. Let’s say our random choice is **$d = 2$**.



---

### Step 2: Discarding the Offset and Partitioning

1. We drop the first $d = 2$ tokens (`"the", "time"`).
2. We then partition the remaining tokens into consecutive chunks of length $n = 5$:


* **Subsequence 1:** `["machine", "by", "h", "g", "wells"]`
* **Subsequence 2:** `["and", "other", "tales", "is", "here"]`



Each of these partitioned subsequences will act as a baseline sequence for our training minibatches.

---

### Step 3: Creating Input and Target Pairs (The Shift)

For language modeling, the objective is to predict the *next* token based on the tokens we have seen so far.

Because of this, the **target (label) sequence is simply the original sequence shifted by one token ($x_{t+1}$)** relative to the input sequence ($\mathbf{x}_t$).

Let's look at how **Subsequence 1** is turned into an input-target pair:

* **Input Sequence ($\mathbf{x}_t$):** `["machine", "by", "h", "g", "wells"]`


* **Target Sequence ($\mathbf{x}_{t+1}$):** `["by", "h", "g", "wells", "and"]` *(shifted forward by one word)*



#### How the network learns from this:

When the model processes the input at each time step, it is trained to predict the next immediate word:

* Given `"machine"`, predict `"by"`
* Given `"machine", "by"`, predict `"h"`
* Given `"machine", "by", "h"`, predict `"g"`
* And so on...

By partitioning the text this way, we can feed continuous, fixed-length streams of data into recurrent or neural language models while introducing random starting points (`d`) each epoch to improve generalization.

In [17]:
@d2l.add_to_class(d2l.TimeMachine)
def __init__(self, batch_size, num_steps, num_train=10000, num_val=5000):
    super(d2l.TimeMachine, self).__init__()
    self.save_hyperparameters()
    corpus, self.vocab = self.build(self._download())
    array = torch.tensor([corpus[i:i+num_steps+1] for i in range(len(corpus)-num_steps)])
    self.X, self.Y = array[:,:-1], array[:,1:]
    

In [18]:
@d2l.add_to_class(d2l.TimeMachine)
def get_dataloader(self, train):
    idx = slice(0, self.num_train) if train else slice(self.num_train, self.num_train+self.num_val)
    return self.get_tensorloader([self.X, self.Y], train, idx)

In [19]:
data = d2l.TimeMachine(batch_size=2, num_steps=10)
for X, Y in data.train_dataloader():
    print('X:', X, '\nY:', Y)
    break

X: tensor([[10, 20,  0, 20, 13, 10, 17, 17,  6, 19],
        [15,  8,  0, 20, 10,  5,  6,  0, 16,  7]]) 
Y: tensor([[20,  0, 20, 13, 10, 17, 17,  6, 19, 20],
        [ 8,  0, 20, 10,  5,  6,  0, 16,  7,  0]])


### ***A little more on Preplexity***

**Perplexity (PPL)** is the standard evaluation metric for probabilistic language models. Mathematically, it measures how well a probability model predicts a sample sequence; intuitively, it quantifies how **"surprised"** or **"confused"** the model is when encountering new text.

A lower perplexity score indicates that the model assigns higher probabilities to the actual words that appear, meaning it understands the underlying joint distribution of the language much better.

Here is the exact mathematical and intuitive breakdown of perplexity.

---

### 1. The Mathematical Definition

Perplexity is formally defined as the **exponentiated average negative log-likelihood** of a sequence.

Given a test sequence of tokens $X = (x_1, x_2, \dots, x_T)$, the perplexity is calculated as:


$$\text{PPL}(X) = \exp\left( -\frac{1}{T} \sum_{t=1}^T \log P(x_t \mid x_1, x_2, \dots, x_{t-1}) \right)$$

If you look closely at the exponent, the term inside the summation is simply the **Cross-Entropy Loss** per token ($\mathcal{L}$):


$$\text{PPL}(X) = \exp(\mathcal{L})$$


Because perplexity is just an exponential transformation of the cross-entropy loss, any model trained via Maximum Likelihood Estimation (which minimizes cross-entropy) is simultaneously minimizing its perplexity.

---

### 2. The Intuitive Interpretation: "The Branching Factor"

Why exponentiate the loss? Because raw cross-entropy loss is measured in natural units (nats) or bits, which are mathematically clean for gradients but hard for humans to interpret. Exponentiating it converts the metric into an intuitive concept known as the **branching factor**.

The branching factor answers this question:

> *At any given step in the sequence, how many equally likely choices does the model think it has to choose from?*

* **PPL = 1:** The model is a crystal ball. Every single prediction is assigned a probability of $1.0$, meaning it is never wrong or surprised.
* **PPL = $\text{Vocabulary Size}$ ($V$):** The model is completely guessing. Its predictions are no better than a uniform random choice across all possible words in the vocabulary.
* **PPL = 10:** When the model is reading a sentence, at each word it hesitates between roughly 10 equally probable candidates.

For example, if a state-of-the-art language model achieves a perplexity of 15 on a text corpus, it means that on average, its uncertainty about the next token is equivalent to choosing uniformly among 15 possible words, regardless of how massive its total vocabulary (e.g., 50,000 words) actually is.

---

### 3. Why Perplexity Matters for Model Comparison

While training loss tells you how well the model is optimizing its parameters on a training set, **perplexity evaluated on a held-out validation set** is the true gold standard for comparing language models.

* It allows apples-to-apples comparisons between different architectures (e.g., comparing an RNN to a Transformer) or different tokenization strategies, provided they are evaluated on the exact same test corpus and vocabulary.